### Lab 9.1 Attention Implementation

In [2]:
import numpy as np

import torch
from torch import nn
import torch.nn.functional as F

1. Complete the following implementation of scaled dot-product attention.   Run the code cell to verify that the output shape is what it should be.

The output values should be 
    
    0.5997, 0.6081, 0.6122, 0.5962, 0.6131

In [9]:
def attention(Q,K,V):
  """
  Computes scaled dot-product attention.

  Compute scores as Q*K^T. 
  Divide by sqrt(d_k).
  Compute softmax on scores along the rows to obtain attention weights.
  Matrix multiply attention weights by values.

  Arguments:
    Q: queries [B,L,d_k]
    K: keys    [B,S,d_k]
    V: values  [B,S,d_v]

  Returns:
    Sequence of context vectors of shape [B,L,d_v]
  """
  # YOUR CODE HERE
  Z = Q@torch.transpose(K, 1, 2)  # [B,L,S]
  d_k = Q.size(-1)
  Z_norm = Z / np.sqrt(d_k)
  sm = F.softmax(Z_norm, dim=2)
  return sm@V

torch.manual_seed(42)

Q = torch.rand(1,5,3)
K = torch.rand(1,5,3)
V = torch.rand(1,5,1)

y = attention(Q,K,V)

y.shape, y

(torch.Size([1, 5, 1]),
 tensor([[[0.5997],
          [0.6081],
          [0.6122],
          [0.5962],
          [0.6131]]]))

2. Complete this implementation of the attention head.  The output should be

```
[[[ 0.4188,  0.1665, -0.6492],
  [ 0.4200,  0.1658, -0.6496],
  [ 0.4140,  0.1691, -0.6468],
  [ 0.4147,  0.1686, -0.6472],
  [ 0.4182,  0.1668, -0.6488]]]
```

In [11]:
class AttentionHead(nn.Module):
    def __init__(self,d_model,d_k):
        super().__init__()
        # create linear projections WQ, WK, WV
        self.WQ = nn.Linear(d_model, d_k)
        self.WK = nn.Linear(d_model, d_k)
        self.WV = nn.Linear(d_model, d_k)

    def forward(self,Q,K,V):
        """ Compute attention head.

            Project the input to queries, keys, and values, and then apply attention.
            Arguments:
                Q: queries [B,L,d_model]
                K: keys    [B,S,d_model]
                V: values  [B,L,d_model]
            Output:
                Context vectors [B,L,d_k]
        """
        # apply linear projections to queries, keys, and values followed by attention
        # YOUR CODE HERE
        Q_proj = self.WQ(Q)
        K_proj = self.WK(K)
        V_proj = self.WV(V)
        return attention(Q_proj, K_proj, V_proj)

torch.manual_seed(42)

Q = torch.rand(1,5,3)
K = torch.rand(1,5,3)
V = torch.rand(1,5,3)

AttentionHead(3,3)(Q,K,V)

tensor([[[ 0.4188,  0.1665, -0.6492],
         [ 0.4200,  0.1658, -0.6496],
         [ 0.4140,  0.1691, -0.6468],
         [ 0.4147,  0.1686, -0.6472],
         [ 0.4182,  0.1668, -0.6488]]], grad_fn=<UnsafeViewBackward0>)